# Развитие игровой индустрии за 2000-2013гг.

- Автор: Алиев Эмил Гадим оглы
- Дата: 10.02.2026

### 0.1. Цели и задачи проекта

Цели: выявить ключевые тренды, определить самые популярные платформы и оценить распределение игр по качеству (оценкам пользователей и критиков)

**Задачи:**
1. Отобрать данные по времени выхода игры за период с 2000 по 2013 год включительно.
2. Категоризовать игры по оценкам пользователей и экспертов, где:
высокая оценка — с оценкой от 8 до 10 и от 80 до 100, включая правые границы интервалов.
средняя оценка — с оценкой от 3 до 8 и от 30 до 80, не включая правые границы интервалов.
низкая оценка — с оценкой от 0 до 3 и от 0 до 30, не включая правые границы интервалов.
3. Выделить топ-7 платформ по количеству игр, выпущенных за весь требуемый период.

### 0.2. Описание данных

Данные содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:  
- Name — название игры.
- Platform — название платформы.
- Year of Release — год выпуска игры.
- Genre — жанр игры.
- NA sales — продажи в Северной Америке (в миллионах проданных копий).
- EU sales — продажи в Европе (в миллионах проданных копий).
- JP sales — продажи в Японии (в миллионах проданных копий).
- Other sales — продажи в других странах (в миллионах проданных копий).
- Critic Score — оценка критиков (от 0 до 100).
- User Score — оценка пользователей (от 0 до 10).
- Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### 0.3. Содержимое проекта

1. [Загрузка и знакомство с данными](#якорь1)
2. [Проверка ошибок в данных и их предобработка](#якорь2)\
    2.1 [Переименование столбцов датафрейма](#якорь2.1)\
    2.2 [Типы данных](#якорь2.2)\
    2.3 [Наличие пропусков в данных](#якорь2.3)\
    2.4 [Явные и неявные дубликаты в данных](#якорь2.4)
3. [Фильтрация данных](#якорь3)
4. [Категоризация данных](#якорь4)
5. [Итоговый вывод](#якорь5)

---

<a id='якорь1'></a>
## 1. Загрузка данных и знакомство с данными



In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/datasets/new_games.csv')

In [3]:
df.head(20)

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.0,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN
6,New Super Mario Bros.,DS,2006.0,Platform,11.28,9.14,6.5,2.88,89.0,8.5,E
7,Wii Play,Wii,2006.0,Misc,13.96,9.18,2.93,2.84,58.0,6.6,E
8,New Super Mario Bros. Wii,Wii,2009.0,Platform,14.44,6.94,4.7,2.24,87.0,8.4,E
9,Duck Hunt,NES,1984.0,Shooter,26.93,0.63,0.28,0.47,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


<br/>
<div class="alert alert-info">
Представленные данные совсем небольшого объёма (1.4 MB). Смысла уменьшать разрядность нет.  
    
11 столбцов, 6 из которых имеют пропуски. нужно будет менять тип данных в столбцах необходимых для исследования. Думаю, что стоит перевести названия столбцов к snake формату и подумать, что сделать с пропусками в столбцах, которые нужны для исследования.  

Для будущего исследования нам точно понадобится заполнить пропуски в столбцах critic_score, user_score. Также поменять тип данных в столбцах user_score, eu_sales и jp_sales. Строки с пропусками в столбце year_of_release можем удалить, т.к. они составляют менее 2% от все данных и сильно не повлияют на результат исследования. Остальные пропуски для исследования не критичны и мы их проигнорируем.
</div>

---
<a id='якорь2'></a>
## 2. Проверка ошибок в данных и их предобработка

<a id='якорь2.1'></a>
### 2.1. Переименование столбцов датафрейма

- Выведите на экран названия всех столбцов датафрейма и проверьте их стиль написания.
- Приведите все столбцы к стилю snake case. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [5]:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


In [6]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [7]:
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')


<a id='якорь2.2'></a>
### 2.2. Типы данных


<br/>
<div class="alert alert-info">
Встречается некорректный тип данных в нужном нам столбце - user score. Предположительная ошибка - человеческий фактор. Преобразуем тип во float64
    </div>

In [8]:
df[['user_score', 'eu_sales', 'jp_sales']] = df[['user_score', 'eu_sales', 'jp_sales']].apply(pd.to_numeric, errors='coerce')
print(df.info())
start_rows = df.shape[0] # количество строк до удаления данных

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  object 
dtypes: float64(7), object(4)
memory usage: 1.4+ MB
None


<br/>
<div class="alert alert-info">
После преобразования типа данных в столбцах eu_sales и jp_sales появились пропуски
        </div>

<a id='якорь2.3'></a>
### 2.3. Наличие пропусков в данных




In [9]:
missing_table = pd.DataFrame({
    'Количество пропусков': df.isnull().sum(),
    'Доля пропусков (%)': (df.isnull().mean() * 100).round(2)
})

missing_table = missing_table.sort_values('Количество пропусков', ascending=False)
missing_table

,Количество пропусков,Доля пропусков (%)
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01
platform,0,0.00
na_sales,0,0.00


<br/>
<div class="alert alert-info">
Пропущенные данные в столбцах в количестве менее 2% мы можем проигнорировать, т.к. оно не существенно и сильно не повлияет на итоговые результаты. Эти данные будем удалять из датасета после проверки, если необходимо.
Данные по рейтингу по всей видимости не заполнялись в полной мере. Возможно  источник, из которого брались данные, не предоставлял их. Удалять их не будем, т.к. их общее количество это половина всех данных. Скорее всего будем выделять в отдельную категорию "без рейтинга".
            </div>

In [10]:
# Проверяем уникальные значения в столбце 'year_of_release'
unique_year_of_release = df['year_of_release'].unique()
print(unique_year_of_release)

[2006. 1985. 2008. 2009. 1996. 1989. 1984. 2005. 1999. 2007. 2010. 2013.
 2004. 1990. 1988. 2002. 2001. 2011. 1998. 2015. 2012. 2014. 1992. 1997.
 1993. 1994. 1982. 2016. 2003. 1986. 2000.   nan 1995. 1991. 1981. 1987.
 1980. 1983.]


<br/>
<div class="alert alert-info">
Диапозон лет 1980–2016 (что является нормальным). Из лишнего тут только nan (пропуски)\
Пропуски, скорее всего, являются технической ошибкой. Заполнять их какими либо значения кроме верных нет никакого смысла, т.к. это исказит данные.
Пропуски в year_of_release не поддаются восстановлению без внешних источников, что к тому же может занять много времени.
Удаляем строки с nan в year_of_release
        </div>

In [11]:
df=df.dropna(subset=['year_of_release'])
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16681 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16681 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16679 non-null  object 
 4   na_sales         16681 non-null  float64
 5   eu_sales         16675 non-null  float64
 6   jp_sales         16677 non-null  float64
 7   other_sales      16681 non-null  float64
 8   critic_score     8085 non-null   float64
 9   user_score       7558 non-null   float64
 10  rating           9901 non-null   object 
dtypes: float64(7), object(4)
memory usage: 1.5+ MB
None


In [12]:
# Проверим пропуски в строках 'name' и 'genre'
missing_name = df[df['name'].isnull()]
missing_genre = df[df['genre'].isnull()]
print(missing_name)
print(missing_genre)


      name platform  year_of_release genre  na_sales  eu_sales  jp_sales  \
661    NaN      GEN           1993.0   NaN      1.78      0.53      0.00   
14439  NaN      GEN           1993.0   NaN      0.00      0.00      0.03   

       other_sales  critic_score  user_score rating  
661           0.08           NaN         NaN    NaN  
14439         0.00           NaN         NaN    NaN  
      name platform  year_of_release genre  na_sales  eu_sales  jp_sales  \
661    NaN      GEN           1993.0   NaN      1.78      0.53      0.00   
14439  NaN      GEN           1993.0   NaN      0.00      0.00      0.03   

       other_sales  critic_score  user_score rating  
661           0.08           NaN         NaN    NaN  
14439         0.00           NaN         NaN    NaN  


In [13]:
# Как видно из данных, пропуски находятся в одинаковых местах (индексы 661 и 14439), что делает их заполнение невозможным. Удаляем данные строки
df = df.dropna(subset=['name'])
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16679 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16679 non-null  object 
 2   year_of_release  16679 non-null  float64
 3   genre            16679 non-null  object 
 4   na_sales         16679 non-null  float64
 5   eu_sales         16673 non-null  float64
 6   jp_sales         16675 non-null  float64
 7   other_sales      16679 non-null  float64
 8   critic_score     8085 non-null   float64
 9   user_score       7558 non-null   float64
 10  rating           9901 non-null   object 
dtypes: float64(7), object(4)
memory usage: 1.5+ MB
None


In [14]:
# Проверяем пропуски в 'eu_sales' и 'jp_sales'
missing_eu_sales = df[df['eu_sales'].isnull()]
missing_jp_sales = df[df['jp_sales'].isnull()]
print(missing_eu_sales)
print(missing_jp_sales)

                                  name platform  year_of_release     genre  \
446                      Rhythm Heaven       DS           2008.0      Misc   
802                        Dead Rising     X360           2006.0    Action   
1131  Prince of Persia: Warrior Within      PS2           2004.0    Action   
1132                         Far Cry 4     XOne           2014.0   Shooter   
1394                   Sonic Advance 3      GBA           2004.0  Platform   
1612                       Ratatouille       DS           2007.0    Action   

      na_sales  eu_sales  jp_sales  other_sales  critic_score  user_score  \
446       0.55       NaN      1.93         0.13          83.0         9.0   
802       1.16       NaN      0.08         0.20          85.0         7.6   
1131      0.54       NaN      0.00         0.22          83.0         8.5   
1132      0.80       NaN      0.01         0.14          82.0         7.5   
1394      0.74       NaN      0.08         0.06          79.0       

In [15]:
# Пропуски находятся в разных местах. Будем заполнять средними значениями по году выпуска и платформе
# Вычисляем средние значения для нужныз нам платформ
mean_eu_DS = df[df['platform'] == 'DS']['eu_sales'].mean()
mean_eu_X360 = df[df['platform'] == 'X360']['eu_sales'].mean()
mean_eu_PS2 = df[df['platform'] == 'PS2']['eu_sales'].mean()
mean_eu_XOne = df[df['platform'] == 'XOne']['eu_sales'].mean()
mean_eu_GBA = df[df['platform'] == 'GBA']['eu_sales'].mean()
mean_jp_X360 = df[df['platform'] == 'X360']['jp_sales'].mean()
mean_jp_DS = df[df['platform'] == 'DS']['jp_sales'].mean()
mean_jp_PSP = df[df['platform'] == 'PSP']['jp_sales'].mean()

In [16]:
# Напишем функцию для заполнения прпоусков в eu_sales
def fill_eu_sales(row):
    if pd.isna(row['eu_sales']):
        if row['platform'] == 'DS':
            return mean_eu_DS
        elif row['platform'] == 'X360':
            return mean_eu_X360
        elif row['platform'] == 'PS2':
            return mean_eu_PS2
        elif row['platform'] == 'XOne':
            return mean_eu_XOne
        elif row['platform'] == 'GBA':
            return mean_eu_GBA
        else:
            return row['eu_sales'] 
    else:
        return row['eu_sales']

# для jp_sales
def fill_jp_sales(row):
    if pd.isna(row['jp_sales']):
        if row['platform'] == 'X360':
            return mean_jp_X360
        elif row['platform'] == 'DS':
            return mean_jp_DS
        elif row['platform'] == 'PSP':
            return mean_jp_PSP
        else:
            return row['jp_sales']
    else:
        return row['jp_sales']

# Пробегаемся по каждой строке и заполняем пропуски
df['eu_sales'] = df.apply(fill_eu_sales, axis=1)
df['jp_sales'] = df.apply(fill_jp_sales, axis=1)

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16679 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16679 non-null  object 
 1   platform         16679 non-null  object 
 2   year_of_release  16679 non-null  float64
 3   genre            16679 non-null  object 
 4   na_sales         16679 non-null  float64
 5   eu_sales         16679 non-null  float64
 6   jp_sales         16679 non-null  float64
 7   other_sales      16679 non-null  float64
 8   critic_score     8085 non-null   float64
 9   user_score       7558 non-null   float64
 10  rating           9901 non-null   object 
dtypes: float64(7), object(4)
memory usage: 1.5+ MB
None


<br/>
<div class="alert alert-info">
Пропуски заполнены, переходим к дубликатам.
Поля с рейтингом не заполняем, т.к. это исказит данные. 
Будем выделять их в отдельную группу "Без рейтинга".
            </div>

<a id='якорь2.4'></a>
### 2.4. Явные и неявные дубликаты в данных


In [17]:
unique_platform = df['platform'].unique()
print(unique_platform)
unique_user_score = df['user_score'].unique()
print(unique_user_score)
unique_critic_score = df['critic_score'].unique()
print(unique_critic_score)
unique_rating = df['rating'].unique()
print(unique_rating)
unique_genre = df['genre'].unique()
print(unique_genre)
# Проверяем данные в столбце platform, чтобы убедиться, что в названии игровых платформ нет неявных дубликатов
# Также проверяем нужные нам столбцы с оценками, жанрами и рейтингом на предмет некорректных данных. Группу с оценками nan будем выделять в отдельную категорию "без оценки"

['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']
[8.  nan 8.3 8.5 6.6 8.4 8.6 7.7 6.3 7.4 8.2 9.  7.9 8.1 8.7 7.1 3.4 5.3
 4.8 3.2 8.9 6.4 7.8 7.5 2.6 7.2 9.2 7.  7.3 4.3 7.6 5.7 5.  9.1 6.5 8.8
 6.9 9.4 6.8 6.1 6.7 5.4 4.  4.9 4.5 9.3 6.2 4.2 6.  3.7 4.1 5.8 5.6 5.5
 4.4 4.6 5.9 3.9 3.1 2.9 5.2 3.3 4.7 5.1 3.5 2.5 1.9 3.  2.7 2.2 2.  9.5
 2.1 3.6 2.8 1.8 3.8 0.  1.6 9.6 2.4 1.7 1.1 0.3 1.5 0.7 1.2 2.3 0.5 1.3
 0.2 0.6 1.4 0.9 1.  9.7]
[76. nan 82. 80. 89. 58. 87. 91. 61. 97. 95. 77. 88. 83. 94. 93. 85. 86.
 98. 96. 90. 84. 73. 74. 78. 92. 71. 72. 68. 62. 49. 67. 81. 66. 56. 79.
 70. 59. 64. 75. 60. 63. 69. 50. 25. 42. 44. 55. 48. 57. 29. 47. 65. 54.
 20. 53. 37. 38. 33. 52. 30. 32. 43. 45. 51. 40. 46. 39. 34. 41. 36. 31.
 27. 35. 26. 19. 28. 23. 24. 21. 17. 13.]
['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']
['Sports' 'Platform' 'Racing' 'Role-Playin

In [18]:
df['name'] = df['name'].str.lower()
df['genre'] = df['genre'].str.lower()
df['rating'] = df['rating'].str.upper().replace('K-A', 'E') # Добавил изменение рейтинга К-А на Е
new_unique_rating = df['rating'].unique()
new_unique_genre = df['genre'].unique()
print(new_unique_rating)
print(new_unique_genre) 
# убираем неявные дубликаты

['E' nan 'M' 'T' 'E10+' 'AO' 'EC' 'RP']
['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']


In [19]:
duplicates = df[df.duplicated()]
print(df.duplicated().sum())
df = df.drop_duplicates()

235


In [20]:
print(f"Дубликатов осталось: {df.duplicated().sum()}")
# Проверяем остались ли дубликаты

Дубликатов осталось: 0


In [21]:
# Были обнаружены явные дубликаты в размере 235 строк. Их тоже удалили
print(df.info())
final_rows = df.shape[0] # записыаем количество строк, после предобработки данных

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16444 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16444 non-null  object 
 1   platform         16444 non-null  object 
 2   year_of_release  16444 non-null  float64
 3   genre            16444 non-null  object 
 4   na_sales         16444 non-null  float64
 5   eu_sales         16444 non-null  float64
 6   jp_sales         16444 non-null  float64
 7   other_sales      16444 non-null  float64
 8   critic_score     7983 non-null   float64
 9   user_score       7463 non-null   float64
 10  rating           9768 non-null   object 
dtypes: float64(7), object(4)
memory usage: 1.5+ MB
None


In [22]:
deleted_rows_count = start_rows - final_rows
deleted_rows_share = round(100 - ((final_rows/start_rows)*100), 2)
print(deleted_rows_count)
print(deleted_rows_share)
# посчитали количество удалённых строк в абсолютном и относительном значениях

512
3.02


<br/>
<div class="alert alert-info">
Данные обработаны и готовы для выполнения поставленных задач.
Итого мы удалили всего 512 строк, что является примерно 3,02% данных. Сильного влияния на результат их отсутствие не окажет
</div>

---
<a id='якорь3'></a>
## 3. Фильтрация данных

Оставим данные за 2000-2013ые года как того требует задача.

In [23]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()
print(df_actual.info())
unique_year_of_release2  = df_actual['year_of_release'].unique()
print(unique_year_of_release2)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12781 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12781 non-null  object 
 1   platform         12781 non-null  object 
 2   year_of_release  12781 non-null  float64
 3   genre            12781 non-null  object 
 4   na_sales         12781 non-null  float64
 5   eu_sales         12781 non-null  float64
 6   jp_sales         12781 non-null  float64
 7   other_sales      12781 non-null  float64
 8   critic_score     7169 non-null   float64
 9   user_score       6483 non-null   float64
 10  rating           8723 non-null   object 
dtypes: float64(7), object(4)
memory usage: 1.2+ MB
None
[2006. 2008. 2009. 2005. 2007. 2010. 2013. 2004. 2002. 2001. 2011. 2012.
 2003. 2000.]


<br/>
<div class="alert alert-info">
В конце проверили корректность фильтрации по нужным нам годам. Всё верно. Идём дальше
            </div>

---
<a id='якорь4'></a>
## 4. Категоризация данных
    
Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8) и низкая оценка (от 0 до 3).

In [24]:
def categorize_user_rating(row):
    if row['user_score'] > 0 and row['user_score'] < 3:
        return "Низкая оценка"
    elif row['user_score'] >= 3 and row['user_score'] < 8:
        return "Средняя оценка"
    elif row['user_score'] >= 8 and row['user_score'] <= 10:
        return "Высокая оценка"
    else:
        return "Без оценки"
df_actual['user_score_category'] = df_actual.apply(categorize_user_rating, axis=1)

Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80) и низкая оценка (от 0 до 30).

In [25]:
def categorize_critic_rating(row):
    if row['critic_score'] > 0 and row['critic_score'] < 30:
        return "Низкая оценка"
    elif row['critic_score'] >= 30 and row['critic_score'] < 80:
        return "Средняя оценка"
    elif row['critic_score'] >= 80 and row['critic_score'] <= 100:
        return "Высокая оценка"
    else:
        return "Без оценки"
df_actual['critic_score_category'] = df_actual.apply(categorize_critic_rating, axis=1)

In [26]:
grouped_user_rating = df_actual.groupby('user_score_category')['name'].count()
print(grouped_user_rating)

user_score_category
Без оценки        6299
Высокая оценка    2286
Низкая оценка      115
Средняя оценка    4081
Name: name, dtype: int64


In [27]:
grouped_critic_rating = df_actual.groupby('critic_score_category')['name'].count()
print(grouped_critic_rating)

critic_score_category
Без оценки        5612
Высокая оценка    1692
Низкая оценка       55
Средняя оценка    5422
Name: name, dtype: int64


Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [28]:
Realesed_games_by_platform = df_actual.groupby('platform')['name'].count()
print(Realesed_games_by_platform) # Вычисляем количество выпущенных игр за заданный период

platform
3DS      300
DC        31
DS      2120
GB        27
GBA      811
GC       542
N64       70
PC       766
PS       274
PS2     2127
PS3     1087
PS4       16
PSP     1180
PSV      134
WS         4
Wii     1275
WiiU      74
X360    1121
XB       803
XOne      19
Name: name, dtype: int64


In [29]:
top_7_platforms = Realesed_games_by_platform.sort_values(ascending=False).head(7)
print(top_7_platforms) # Сортируем, делая топ7

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: name, dtype: int64


---
<a id='якорь5'></a>
## 5. Итоговый вывод


Была проделана большая работа по предобработке, фильтрации и категоризации данных.  
Датасет не позволил в полной мере категоризовать данные по оценками зрителей и критиков из-за отсутствия половины данных по данным значениям.\
Но из того что есть, можно сделать вывод, что:  
* Самая популярная категория оценок у пользователей и у критиков это "средняя оценка", что может говорить о сдержанности в выставлении оценок
* Выходило больше хороших игр, нежели плохих. Очень много высоких и средних оценок в сравнении с низкими.
  
Самыми популярными платформами у издателей игр были:
* **PS2** - 2127 игр,
* **DS** - 2120 игр,
* **Wii** - 1275 игр,
* **PSP** - 1180 игр,
* **X360** - 1121 игр
  
В исходный датасет были добавлены такие поля, как:  
    * user_score_category - для категорезации по оценкам пользователей  
    * critic_score_category - для категоризации по оценкам критиков
